# Dashboard tables freshness check

Checks table availability, max business date, target date readiness, and key join coverage.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None

In [ ]:
# Parameters
# Ставь дату, на которую проверяем готовность витрины.
target_date = '2026-05-31'
mem_limit = '8g'

# SLA для технической загрузки: сколько часов максимум прошло с последнего commit/insert.
tech_freshness_hours = 36

# Для скорости: metadata refresh выключен по умолчанию.
use_metadata_refresh = False

# Отсекаем мусорные даты (например, 2079 год).
min_valid_date = '2020-01-01'

# Проверяем только ключевые таблицы.
# date_expr взят из фактических колонок по фото.
tables_cfg = [
    {'table': 'ods_alpha.scd1_agreements', 'date_expr': "to_date(cast(d_valid_from as timestamp))", 'critical': 1,
     'key_exprs': ["n_agr is null", "n_cmp_client is null", "c_agr_number is null"]},
    {'table': 'ods_alpha.scd1_companies', 'date_expr': "coalesce(cast(d_ins as date), to_date(cast(ods_insert_ts as timestamp)))", 'critical': 1,
     'key_exprs': ["n_cmp is null", "c_inn is null or trim(c_inn) = ''"]},
    {'table': 'ods_alpha.scd1_merchants', 'date_expr': "to_date(cast(d_ins as timestamp))", 'critical': 1,
     'key_exprs': ["c_nmrc is null or trim(c_nmrc) = ''", "n_cmp is null", "n_mcc is null"]},
    {'table': 'ods_alpha.scd1_pos_terminals', 'date_expr': "to_date(cast(d_ins as timestamp))", 'critical': 1,
     'key_exprs': ["c_nter is null or trim(c_nter) = ''", "c_nmrc is null or trim(c_nmrc) = ''"]},
    {'table': 'ods_alpha.scd1_trx', 'date_expr': "to_date(cast(d_trx_orig as timestamp))", 'critical': 1,
     'where_expr': "coalesce(ods_deleted_flg, '0') <> '1'",
     'key_exprs': ["n_trx is null", "c_nter is null or trim(c_nter) = ''", "n_amt_src is null"]},
    {'table': 'ods_alpha.scd1_trx_acq', 'date_expr': "to_date(cast(d_trx_local as timestamp))", 'critical': 1,
     'key_exprs': ["n_trx is null", "n_agr is null", "n_amt_tax is null"]},
    {'table': 'ods_alpha.scd1_trx_int', 'date_expr': "coalesce(to_date(cast(d_int_setl as timestamp)), to_date(cast(d_net_setl as timestamp)))", 'critical': 1,
     'key_exprs': ["n_trx is null", "n_amt_fee is null"]},
]

print('target_date =', target_date)
print('mem_limit =', mem_limit)
print('tech_freshness_hours =', tech_freshness_hours)
print('key_tables =', len(tables_cfg))

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    if use_metadata_refresh:
        for cfg in tables_cfg:
            try:
                imp.execute(f"invalidate metadata {cfg['table']}")
            except Exception:
                pass

print('Impala connected')

In [ ]:
rows = []
for cfg in tables_cfg:
    table = cfg['table']
    date_expr = cfg['date_expr']
    critical = cfg['critical']
    key_exprs = cfg.get('key_exprs', [])
    where_expr = cfg.get('where_expr')

    try:
        # Быстрый smoke-check доступности таблицы (вместо тяжелого count(*))
        sql_exists = f"select 1 as ok from {table} limit 1"
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            _ = imp.fetch(sql_exists)

        where_sql = f"where {where_expr}" if where_expr else ""

        # Одним агрегатом собираем business freshness + tech freshness + null-rate ключевых полей.
        key_null_aggs = []
        for idx, expr in enumerate(key_exprs, 1):
            key_null_aggs.append(f"sum(case when {expr} then 1 else 0 end) as key_null_{idx}")
        key_null_sql = ",\n              ".join(key_null_aggs) if key_null_aggs else "cast(0 as bigint) as key_null_1"

        sql_main = f"""
        select
          count(*) as row_cnt,
          max(case when {date_expr} between cast('{min_valid_date}' as date) and date_add(current_date(), 1)
                   then {date_expr} end) as max_business_date,
          sum(case when {date_expr} = cast('{target_date}' as date) then 1 else 0 end) as target_rows,
          max(to_timestamp(ods_insert_ts, 'yyyy-MM-dd HH:mm:ss')) as max_ods_insert_ts,
          max(to_timestamp(ods_commit_ts, 'yyyy-MM-dd HH:mm:ss')) as max_ods_commit_ts,
          {key_null_sql}
        from {table}
        {where_sql}
        """

        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            main_df = imp.fetch(sql_main)

        row_cnt = int(main_df.loc[0, 'row_cnt']) if len(main_df) else 0
        max_business_date = main_df.loc[0, 'max_business_date'] if len(main_df) else None
        target_rows = int(main_df.loc[0, 'target_rows']) if len(main_df) and pd.notna(main_df.loc[0, 'target_rows']) else 0
        max_ods_insert_ts = main_df.loc[0, 'max_ods_insert_ts'] if len(main_df) else None
        max_ods_commit_ts = main_df.loc[0, 'max_ods_commit_ts'] if len(main_df) else None

        key_null_total = 0
        for i in range(1, len(key_exprs) + 1):
            c = f'key_null_{i}'
            key_null_total += int(main_df.loc[0, c]) if len(main_df) and pd.notna(main_df.loc[0, c]) else 0
        key_null_rate_pct = round(100.0 * key_null_total / row_cnt, 4) if row_cnt > 0 else None

        # technical freshness in hours
        latest_tech_ts = max_ods_commit_ts if pd.notna(max_ods_commit_ts) else max_ods_insert_ts
        tech_delay_hours = None
        tech_stale = None
        if pd.notna(latest_tech_ts):
            now_ts = pd.Timestamp.now()
            latest_tech_ts = pd.to_datetime(latest_tech_ts)
            tech_delay_hours = round((now_ts - latest_tech_ts).total_seconds() / 3600.0, 2)
            tech_stale = tech_delay_hours > tech_freshness_hours

        # Светофор качества: business freshness + tech freshness + null-rate
        if target_rows <= 0:
            status = 'lagging'
        elif tech_stale is True:
            status = 'tech_stale'
        elif key_null_rate_pct is not None and key_null_rate_pct > 5:
            status = 'dq_warning'
        else:
            status = 'ready'

        rows.append({
            'table': table,
            'critical': critical,
            'row_cnt': row_cnt,
            'date_expr': date_expr,
            'max_business_date': max_business_date,
            'target_rows': target_rows,
            'max_ods_insert_ts': max_ods_insert_ts,
            'max_ods_commit_ts': max_ods_commit_ts,
            'tech_delay_hours': tech_delay_hours,
            'tech_stale': tech_stale,
            'key_null_total': key_null_total,
            'key_null_rate_pct': key_null_rate_pct,
            'status': status,
            'error': None
        })

    except Exception as e:
        rows.append({
            'table': table,
            'critical': critical,
            'row_cnt': None,
            'date_expr': date_expr,
            'max_business_date': None,
            'target_rows': None,
            'max_ods_insert_ts': None,
            'max_ods_commit_ts': None,
            'tech_delay_hours': None,
            'tech_stale': None,
            'key_null_total': None,
            'key_null_rate_pct': None,
            'status': 'error',
            'error': str(e)[:500]
        })

freshness_summary = pd.DataFrame(rows)
display(freshness_summary.sort_values(['critical', 'status', 'table'], ascending=[False, True, True]))
print('critical_not_ready =', len(freshness_summary[(freshness_summary['critical'] == 1) & (freshness_summary['status'].isin(['lagging', 'tech_stale', 'dq_warning', 'error']))]))

In [ ]:
# trx -> acq/int coverage on target_date (оптимизировано)
# Берем ограниченную выборку trx за дату, чтобы не упереться в память.
sql_cov = f"""
with trx_day as (
  select cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  where to_date(cast(t.d_trx_orig as timestamp)) = cast('{target_date}' as date)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
  limit 200000
),
acq as (
  select distinct cast(n_trx as string) as n_trx
  from ods_alpha.scd1_trx_acq
),
ints as (
  select distinct cast(n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int
)
select
  count(*) as trx_sample_cnt,
  sum(case when a.n_trx is not null then 1 else 0 end) as trx_with_acq,
  sum(case when i.n_trx is not null then 1 else 0 end) as trx_with_int,
  round(100.0 * sum(case when a.n_trx is not null then 1 else 0 end) / nullif(count(*),0), 2) as acq_coverage_pct,
  round(100.0 * sum(case when i.n_trx is not null then 1 else 0 end) / nullif(count(*),0), 2) as int_coverage_pct
from trx_day t
left join acq a on a.n_trx = t.n_trx
left join ints i on i.n_trx = t.n_trx
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    coverage_df = imp.fetch(sql_cov)

display(coverage_df)

In [ ]:
# Simple business readiness verdict
critical_bad = freshness_summary[(freshness_summary['critical'] == 1) & (freshness_summary['status'].isin(['lagging', 'tech_stale', 'dq_warning', 'error']))]
if len(critical_bad) == 0:
    print('READY: all critical sources look ready for target_date')
else:
    print('NOT READY: critical sources have issues')
    display(critical_bad[['table', 'status', 'max_business_date', 'target_rows', 'max_ods_commit_ts', 'max_ods_insert_ts', 'tech_delay_hours', 'key_null_rate_pct', 'error']])